# 04 ? Collect and score saved answers
I use the frozen tolerant solution-move mention scorer. Claude bulk collection is now explicitly authorized, selecting Claude only and reusing the completed 100-puzzle pilot. I save each answer and pause at provider limits. The original pilot checkpoint remains preserved.


## 1. My controls

`setup` checks locally. `pilot` collects up to 100 training answers per model. `bulk` continues to 1,000. `WAIT_FOR_QUOTA=True` waits and resumes while the kernel/computer stays running; otherwise each provider saves and pauses independently.

In [1]:
RUN_MODE = "bulk"
WAIT_FOR_QUOTA = True
SELECTED_MODELS = ["gpt_oss"]


In [2]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
import os, json, hashlib, re, time, math, uuid, warnings, platform
import requests, chess, nbformat
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'experiments/stage_index.json').exists())
CFG = ROOT/'experiments/one_move_v1'
OUT = ROOT/'data/experiments/one_move_v1'; OUT.mkdir(exist_ok=True)
CONFIG = json.loads((CFG/'config.json').read_text())
PLAN = json.loads((CFG/'analysis_plan.json').read_text())
PROMPT = (CFG/'prompt.txt').read_text()
assert RUN_MODE in ['setup', 'pilot', 'bulk'] and isinstance(WAIT_FOR_QUOTA, bool)
PACIFIC = ZoneInfo('America/Los_Angeles')
SECRETS = []  # I load credentials only if I explicitly enable live collection.

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def redact(value):
    if isinstance(value, str):
        for secret in SECRETS: value = value.replace(secret, '[REDACTED]')
        return value
    if isinstance(value, dict): return {k: redact(v) for k, v in value.items()}
    if isinstance(value, list): return [redact(v) for v in value]
    return value

def save(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(redact(value), indent=2, ensure_ascii=False)+'\n', encoding='utf-8')
    os.replace(temporary, path)

def records(path):
    path = Path(path)
    if not path.exists(): return []
    text = path.read_text(encoding='utf-8'); lines = text.splitlines(); rows = []
    for i, line in enumerate(lines):
        if not line.strip(): continue
        try: rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i == len(lines)-1 and not text.endswith('\n'):
                warnings.warn(f'I found an incomplete final log record in {path.name}; it is not a completed response.')
            else: raise
    return rows

def append(path, row):
    path = Path(path)
    # I repair only an incomplete last line after a crash, under the provider file lock.
    if path.exists():
        raw = path.read_bytes()
        if raw and not raw.endswith(b'\n'):
            try: json.loads(raw.splitlines()[-1]); prefix = raw+b'\n'
            except json.JSONDecodeError: prefix = raw[:raw.rfind(b'\n')+1]
            path.write_bytes(prefix)
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(redact(row), ensure_ascii=False)+'\n'); stream.flush(); os.fsync(stream.fileno())

@contextmanager
def provider_lock(folder):
    # OS locks are released if the kernel exits, so stale PID files cannot block a resume.
    with (folder/'collection.lock').open('a+b') as handle:
        if handle.tell() == 0: handle.write(b'0'); handle.flush()
        handle.seek(0)
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        try: yield
        finally:
            handle.seek(0)
            if os.name == 'nt': msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else: fcntl.flock(handle.fileno(), fcntl.LOCK_UN)

def code_hash(name):
    n = nbformat.read(ROOT/'notebooks'/name, as_version=4)
    code = '\n\n'.join(c.source for c in n.cells if c.cell_type == 'code' and 'run_control' not in c.metadata.get('tags', []))
    return hashlib.sha256(code.encode()).hexdigest()

manifest = {'version': CONFIG['version'], 'config_sha256': sha(CFG/'config.json'),
            'plan_sha256': sha(CFG/'analysis_plan.json'), 'final_plan_sha256': sha(CFG/'final_analysis_plan.json'), 'prompt_sha256': sha(CFG/'prompt.txt'),
            'collection_code_sha256': code_hash('04_collect_answers.ipynb'),
            'analysis_code_sha256': code_hash('06_final_analysis.ipynb'),
            'preparation_code_sha256': code_hash('03_prepare_study.ipynb'),
            'early_code_sha256': code_hash('05_early_results.ipynb')}
if (OUT/'manifest.json').exists():
    assert manifest == json.loads((OUT/'manifest.json').read_text()), 'Frozen study changed; create a separate version.'
else: save(OUT/'manifest.json', manifest)

# I keep the original two-model settings frozen; Claude is an explicit pilot amendment.
CLAUDE_AMENDMENT = json.loads((CFG/'claude_pilot.json').read_text())
assert sha(CFG/'claude_pilot.json') == json.loads((OUT/'claude_amendment_binding.json').read_text())['config_sha256']
CONFIG['models']['claude_haiku'] = CLAUDE_AMENDMENT['model_spec']
assert SELECTED_MODELS and set(SELECTED_MODELS) <= set(CONFIG['models'])
if 'claude_haiku' in SELECTED_MODELS:
    assert RUN_MODE != 'bulk' or CLAUDE_AMENDMENT.get('bulk_authorized') is True, 'Claude bulk collection requires documented approval.'

# I protect historical raw responses and fitted models. Viewer-facing Markdown is editable.
if not (OUT/'historical_hashes.json').exists():
    protected = {p.relative_to(ROOT).as_posix(): sha(p)
                 for parent in ['data/experiments', 'data/analysis']
                 for p in (ROOT/parent).rglob('*') if p.is_file() and OUT not in p.parents}
    save(OUT/'historical_hashes.json', protected)
else: protected = json.loads((OUT/'historical_hashes.json').read_text())
assert all(sha(ROOT/p) == digest for p, digest in protected.items())
print('I verified my frozen protocol and protected earlier response/model files.')
def score_strict(board,task,response,expected=None):
    choice=response['choices'][0]
    if choice.get('finish_reason')=='length':return 'generation_limit',None,False
    if choice.get('finish_reason')!='stop':return 'nonstop_termination',None,False
    answer=choice.get('message',{}).get('content')
    if not isinstance(answer,str):return 'unparseable',None,False
    answer=answer.strip()
    if task=='side':
        if answer not in ['white','black']:return 'unparseable',answer,False
        return ('success' if answer==('white' if board.turn else 'black') else 'incorrect_side'),answer,True
    if re.fullmatch('[a-h][1-8][a-h][1-8][qrbn]?',answer):
        move=chess.Move.from_uci(answer)
        if move not in board.legal_moves:return 'illegal_move',answer,False
    else:
        try:
            move=board.parse_san(answer)
            if move not in board.legal_moves:return 'illegal_move',answer,False
            canonical=board.san(move)
            normalized=answer.replace('0-0-0','O-O-O').replace('0-0','O-O')
            if normalized not in {canonical,canonical.rstrip('+#')}:return 'unparseable',answer,False
        except ValueError:return 'unparseable',answer,False
    answer=move.uci()
    if task=='mate':
        copy=board.copy();copy.push(move)
        correct=copy.is_checkmate()
    else:correct=task=='legal' or answer==expected
    return ('success' if correct else 'legal_incorrect'),answer,True
SCORING_RULE=json.loads((CFG/'scoring_amendment.json').read_text())
assert sha(CFG/'scoring_amendment.json')==json.loads((OUT/'scoring_binding.json').read_text())['config_sha256']
SCORING_VERSION=SCORING_RULE['revision']

# I recognize standalone move tokens in the visible answer, never hidden thinking.
UCI_MENTION=re.compile(r'(?<![A-Za-z0-9_])([a-h][1-8](?:\s*[-??]\s*)?[a-h][1-8][qrbn]?)(?![A-Za-z0-9_])',re.I)
SAN_MENTION=re.compile(r'(?<![A-Za-z0-9_])((?:[Oo0]-[Oo0](?:-[Oo0])?|[KQRBNkqrbn]?[a-h]?[1-8]?x?[a-h][1-8](?:=?[QRBNqrbn])?)[+#]?)(?![A-Za-z0-9_])')

def mentioned_moves(board,text):
    if not isinstance(text,str):return []
    found=[]
    for match in UCI_MENTION.finditer(text):
        token=re.sub(r'[\s??-]','',match.group(1)).lower()
        try:move=chess.Move.from_uci(token)
        except ValueError:continue
        if move in board.legal_moves:found.append((match.start(),move.uci()))
    for match in SAN_MENTION.finditer(text):
        token=match.group(1).replace('0','O').replace('o','O')
        candidates=[token]
        if token and token[0] in 'kqrbn':candidates.append(token[0].upper()+token[1:])
        for candidate in candidates:
            try:move=board.parse_san(candidate)
            except ValueError:continue
            if move in board.legal_moves:
                found.append((match.start(),move.uci()));break
    return list(dict.fromkeys(move for _,move in sorted(found)))

def score(board,task,response,expected=None):
    if task!='evaluation':return score_strict(board,task,response,expected)
    choice=response.get('choices',[{}])[0]
    moves=mentioned_moves(board,choice.get('message',{}).get('content'))
    if expected in moves:return 'success',expected,True
    if choice.get('finish_reason')!='stop':return score_strict(board,task,response,expected)
    if moves:return 'legal_incorrect',moves[0],True
    return score_strict(board,task,response,expected)

def seconds(value):
    try:return float(value)
    except (ValueError,TypeError):
        pairs=re.findall(r'([0-9.]+)(ms|s|m|h)',str(value))
        if pairs:return sum(float(n)*{'ms':.001,'s':1,'m':60,'h':3600}[unit] for n,unit in pairs)
        try:return max(0,(datetime.strptime(value,'%a, %d %b %Y %H:%M:%S GMT').replace(tzinfo=timezone.utc)-datetime.now(timezone.utc)).total_seconds())
        except (ValueError,TypeError):return None

def prompt_for(board, task='evaluation'):
    pieces = '\n'.join(('White' if color else 'Black')+' pieces: '+', '.join(
        f'{chess.piece_name(p.piece_type)} {chess.square_name(s)}'
        for s, p in sorted(board.piece_map().items()) if p.color == color)
        for color in [chess.WHITE, chess.BLACK])
    text = PROMPT.format(side='White' if board.turn else 'Black', fen=board.fen(),
                         pieces=pieces, choices=', '.join(sorted(m.uci() for m in board.legal_moves)))
    if task == 'side': return f'Which side moves next in this FEN? {board.fen()}\n'+pieces+'\nReturn only white or black in the final answer.'
    if task == 'mate': return text.replace('Choose the best next move', 'Choose a move that delivers checkmate immediately')
    if task == 'legal': return text.replace('Choose the best next move', 'Choose any legal next move')
    return text

def request_body(spec, board, task='evaluation'):
    text = prompt_for(board, task)
    if spec['provider'] == 'anthropic':
        return {'model': spec['model'], 'max_tokens': spec['max_completion_tokens'],
                'thinking': {'type': spec['thinking_type']}, 'output_config': {'effort': spec['reasoning_effort']},
                'messages': [{'role': 'user', 'content': text}]}
    if spec['provider'] == 'groq':
        return {'model': spec['model'], 'temperature': spec['temperature'], 'seed': spec['seed'],
                'reasoning_effort': spec['reasoning_effort'], 'max_completion_tokens': spec['max_completion_tokens'],
                'messages': [{'role': 'user', 'content': text}]}
    return {'contents': [{'role': 'user', 'parts': [{'text': text}]}],
            'generationConfig': {'temperature': spec['temperature'], 'topP': spec['top_p'], 'seed': spec['seed'],
                'maxOutputTokens': spec['max_completion_tokens'],
                'thinkingConfig': {'thinkingLevel': spec['thinking_level'], 'includeThoughts': False}}}

def normalized(spec, raw):
    if spec['provider'] == 'anthropic':
        content = ''.join(p.get('text', '') for p in raw.get('content', []) if p.get('type') == 'text')
        reason = raw.get('stop_reason', 'unknown')
        return {'choices': [{'message': {'content': content}, 'finish_reason': {'end_turn': 'stop', 'max_tokens': 'length'}.get(reason, reason)}]}
    if spec['provider'] == 'groq': return raw
    candidates = raw.get('candidates', [])
    if not candidates: return {'choices': [{'message': {'content': ''}, 'finish_reason': 'blocked_or_empty'}]}
    candidate = candidates[0]; reason = candidate.get('finishReason', 'UNKNOWN')
    content = ''.join(p.get('text', '') for p in candidate.get('content', {}).get('parts', []) if not p.get('thought', False))
    return {'choices': [{'message': {'content': content}, 'finish_reason': {'STOP': 'stop', 'MAX_TOKENS': 'length'}.get(reason, reason.lower())}]}

start = chess.Board(); after = start.copy(); after.push_uci('e2e4')
mate = chess.Board('7k/8/5KQ1/8/8/8/8/8 w - - 0 1'); mirror = mate.transform(chess.flip_horizontal)
FIXTURES = [('mate_a_white', mate, 'mate'), ('side_white', start, 'side'), ('legal_white', start, 'legal'),
            ('side_black', after, 'side'), ('legal_black', after, 'legal'), ('mate_a_black', mate.mirror(), 'mate'),
            ('mate_b_white', mirror, 'mate'), ('mate_b_black', mirror.mirror(), 'mate')]


I verified my frozen protocol and protected earlier response/model files.


In [3]:
prepared=json.loads((OUT/'preparation.json').read_text())
sample_file=OUT/'puzzles.csv'
assert sha(sample_file)==prepared['sample_sha256']
assert sha(OUT/'analysis_features.csv')==prepared['feature_sha256']
assert sha(OUT/'feature_recipe.json')==prepared['recipe_sha256']
puzzles=pd.read_csv(sample_file,dtype={'puzzle_id':str})
assert len(puzzles)==1000 and puzzles.total_solver_moves.eq(1).all()
assert puzzles.loc[puzzles.pilot,'split'].eq('train').all() and int(puzzles.pilot.sum())==100
print('I verified the frozen 1,000-case sample; the pilot uses 100 training cases only.')


I verified the frozen 1,000-case sample; the pilot uses 100 training cases only.


## 2. My resumable collector and offline checks

I retain historical usage for reporting and minute pacing. Locally summed daily usage never stops collection; provider headers and HTTP 429 evidence determine actual quota pauses. The scorer matches the entire final UCI or normalized canonical SAN answer to the official move. Capability checks are separate from research accuracy.

In [4]:
def epoch(row):
    if row.get('finished_epoch') is not None: return float(row['finished_epoch'])
    return datetime.fromisoformat(row['timestamp_utc'].replace('Z', '+00:00')).timestamp()

def log_model(row):
    return row.get('model') or row.get('request', {}).get('model') or row.get('response', {}).get('modelVersion') or row.get('response', {}).get('model')

def usage_tokens(spec, row):
    raw = row.get('response', {})
    if spec['provider'] == 'anthropic':
        usage = raw.get('usage', {})
        return sum(int(usage.get(k, 0) or 0) for k in ['input_tokens','output_tokens','cache_read_input_tokens','cache_creation_input_tokens'])
    usage = raw.get('usage', {}) if spec['provider'] == 'groq' else raw.get('usageMetadata', {})
    return int(usage.get('total_tokens', usage.get('totalTokenCount', 0)) or 0)

def next_pacific_reset(now):
    local = datetime.fromtimestamp(now, PACIFIC)
    return datetime.combine(local.date()+timedelta(days=1), datetime.min.time(), PACIFIC).timestamp()+5


def rate_limit_details(spec, log):
    """I classify only provider evidence, not locally estimated daily usage."""
    error=log.get('provider_error',{});error=error if isinstance(error,dict) else {'message':str(error)}
    headers=log.get('rate_limits',{});message=json.dumps(error).lower()
    details=error.get('details',[])
    violations=[v for d in details if isinstance(d,dict) for v in d.get('violations',[])]
    zero=any(str(v)=='0' for k,v in headers.items() if 'limit' in k and 'remaining' not in k)
    zero=zero or any(str(v.get('quotaValue'))=='0' for v in violations)
    if zero or 'request too large' in message:
        return {'scope':'access','review':True,'wait':0,'source':'provider_error','reason':'Provider reported zero allowance or request-size restriction; inspect raw error'}
    if 'monthly' in message or 'permonth' in message or 'per month' in message:
        return {'scope':'monthly','review':True,'wait':0,'source':'provider_error','reason':'Provider monthly quota; account review required'}
    delays=[]
    header_delay=seconds(headers.get('retry-after'))
    if header_delay is not None:delays.append(header_delay)
    for d in details:
        if isinstance(d,dict) and 'retryDelay' in d:
            value=seconds(d['retryDelay'])
            if value is not None:delays.append(value)
    match=re.search(r'try again in\s+((?:[0-9]+(?:\.[0-9]+)?\s*(?:ms|s|m|h)\s*)+)',str(error.get('message','')).lower())
    if match:
        value=seconds(match.group(1))
        if value is not None:delays.append(value)
    daily=bool(re.search(r'\b(?:tpd|rpd)\b|per[_ -]?day|daily',message))
    explicit=max(delays) if delays else None
    if daily:
        if spec['provider']=='google':
            wait=max(explicit or 0,next_pacific_reset(time.time())-time.time());source='documented_midnight_pacific_reset'
        elif explicit is not None:wait=explicit+2;source='provider_retry_instruction'
        else:wait=86400;source='fallback_24h_estimate_no_provider_retry_time'
        return {'scope':'daily','review':False,'wait':wait,'source':source,'reason':'Provider daily quota exhausted'}
    minute=bool(re.search(r'\b(?:rpm|tpm|itpm|otpm)\b|per[_ -]?minute',message))
    return {'scope':'minute' if minute else 'rate_limit','review':False,'wait':(explicit if explicit is not None else 60)+2,
            'source':'provider_retry_instruction' if explicit is not None else 'fallback_short_backoff','reason':'Provider minute rate limit' if minute else 'Provider rate limit'}

class Collector:
    def __init__(self, name, spec):
        self.name, self.spec = name, spec
        self.folder = OUT/name; self.folder.mkdir(exist_ok=True)
        self.log_path = self.folder/'requests.jsonl'; self.outcome_path = self.folder/'outcomes.jsonl'
        self.session = requests.Session()
        self.old_logs = []
        for path in (ROOT/'data/experiments').rglob('requests.jsonl'):
            if OUT in path.parents: continue
            self.old_logs.extend(r for r in records(path) if log_model(r) == spec['model'])
    def connect(self):
        key = (ROOT/self.spec['key_file']).read_text(encoding='utf-8-sig').strip()
        if self.spec['provider'] == 'anthropic':
            self.session.headers.update({'x-api-key': key, 'anthropic-version': self.spec['api_version']})
            url = self.spec['base_url']+'/models/'+self.spec['model']
        elif self.spec['provider'] == 'groq':
            self.session.headers['Authorization'] = 'Bearer '+key; url = self.spec['base_url']+'/models'
        else:
            self.session.headers['x-goog-api-key'] = key; url = self.spec['base_url']+'/models/'+self.spec['model']
        del key
        try:
            response = self.session.get(url, timeout=30)
            available = response.ok and (self.spec['provider'] in ['google','anthropic'] or self.spec['model'] in {m['id'] for m in response.json().get('data', [])})
            save(self.folder/'access.json', {'available': bool(available), 'http_status': response.status_code})
            if available: save(self.folder/'provider_metadata.json', response.json())
            return bool(available)
        except (requests.RequestException, ValueError) as error:
            save(self.folder/'access.json', {'available': False, 'error_type': type(error).__name__}); return False
    def latest(self): return {r['case_id']: r for r in records(self.outcome_path)}
    def history(self):
        merged = {}
        for row in self.old_logs+records(self.log_path):
            merged[(row.get('trial_id'), row.get('retry'), row.get('timestamp_utc'))] = row
        return list(merged.values())
    def quota_wait(self, body, now=None):
        now=time.time() if now is None else now;spec=self.spec;history=self.history()
        path=self.folder/'pause.json';pause=json.loads(path.read_text()) if path.exists() else {}
        if pause.get('requires_review'):return 0,pause.get('reason','Access review required'),True
        # Legacy locally estimated daily stops must not block an actual provider attempt.
        local_reasons={'Conservative rolling 24-hour budget','Conservative project-day request budget'}
        wait=0 if pause.get('reason') in local_reasons else max(0,float(pause.get('resume_epoch',0))-now)
        if wait and pause.get('scope')=='daily':return wait,'Provider daily quota (saved)',False
        reason='Saved provider retry time' if wait else None
        if history:
            last=max(history,key=epoch)
            pacing=max(0,spec['minimum_pause_seconds']-(now-epoch(last)))
            if pacing>wait:wait,reason=pacing,'Request pacing'
            if spec['provider']=='groq':
                headers=last.get('rate_limits',{})
                if str(headers.get('x-ratelimit-remaining-requests'))=='0':
                    reset=epoch(last)+(seconds(headers.get('x-ratelimit-reset-requests')) or 86400)+2
                    if reset>now:return reset-now,'Provider daily request allowance exhausted (header)',False
                estimate=math.ceil(len(body['messages'][0]['content'])/2.5)+128
                reservation=spec['max_completion_tokens']+estimate
                allowance=float(headers.get('x-ratelimit-limit-tokens',spec['published_limits_reference']['tpm']))
                if reservation>allowance:return 0,'Request reservation exceeds per-minute allowance',True
                remaining=headers.get('x-ratelimit-remaining-tokens')
                if remaining is not None and float(remaining)<reservation:
                    reset=max(0,(seconds(headers.get('x-ratelimit-reset-tokens')) or 60)-(now-epoch(last))+2)
                    if reset>wait:wait,reason=reset,'Groq minute-token pacing'
        # I do not stop for locally summed daily tokens, cached tokens or request totals.
        return wait,reason,False
    def defer(self, reason, wait=0, review=False, scope=None, retry_source=None):
        now=time.time()
        row={'status':'needs_review' if review else 'paused_for_quota','reason':reason,'scope':scope,
             'requires_review':review,'resume_epoch':now+wait,'resume_utc':datetime.fromtimestamp(now+wait,timezone.utc).isoformat(),
             'retry_source':retry_source}
        save(self.folder/'pause.json',row)
        return row
    def completed_row(self, log, board, task, expected):
        raw = log['response']; value = normalized(self.spec, raw)
        if not value.get('choices'):
            value = {'choices': [{'message': {'content': ''}, 'finish_reason': 'blocked_or_empty'}]}
        category, answer, usable = score(board, task, value, expected)
        row = {'case_id': log['case_id'], 'puzzle_id': log['puzzle_id'], 'task': task, 'model_key': self.name,
               'trial_id': log['trial_id'], 'status': 'completed', 'failure_type': category,
               'answer': answer, 'usable_answer': bool(usable), 'success': int(category == 'success'),
               'returned_model': raw.get('model', raw.get('modelVersion')), 'total_tokens': usage_tokens(self.spec, log),
               'scoring_version': SCORING_VERSION if task=='evaluation' else 'strict_capability'}
        append(self.outcome_path, row)
        return row
    def run_case(self, puzzle_id, board, task='evaluation', expected=None):
        case_id = task+':'+puzzle_id; body = request_body(self.spec, board, task)
        cached = self.latest().get(case_id)
        if cached and cached['status'] == 'completed': return cached
        # I replay a durably saved response if a crash happened before its outcome was written.
        successful = [r for r in records(self.log_path) if r['case_id'] == case_id and 'response' in r]
        if successful:
            log = successful[-1]; assert log['request'] == body
            return self.completed_row(log, board, task, expected)
        trial = str(uuid.uuid4())
        for retry in range(5):
            wait, reason, review = self.quota_wait(body)
            if review: return self.defer(reason, review=True)
            if reason and reason.startswith('Provider daily'): return self.defer(reason, wait, scope='daily', retry_source='provider_saved_or_header')
            if wait > 120: return self.defer(reason, wait)
            while wait > 0:
                time.sleep(min(30, wait)); wait, reason, review = self.quota_wait(body)
                if review: return self.defer(reason, review=True)
                if reason and reason.startswith('Provider daily'): return self.defer(reason, wait, scope='daily', retry_source='provider_saved_or_header')
                if wait > 120: return self.defer(reason, wait)
            spec = self.spec; http = None; retry_wait = None
            url = spec['base_url']+('/messages' if spec['provider'] == 'anthropic' else '/chat/completions' if spec['provider'] == 'groq' else '/models/'+spec['model']+':generateContent')
            log = {'trial_id': trial, 'case_id': case_id, 'puzzle_id': puzzle_id, 'task': task,
                   'model': spec['model'], 'provider': spec['provider'], 'retry': retry,
                   'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'request': body}
            try:
                response = self.session.post(url, json=body, timeout=(15,240)); http = response.status_code
                log['http_status'] = http
                log['rate_limits'] = {k.lower(): v for k, v in response.headers.items() if k.lower().startswith(('x-ratelimit','anthropic-ratelimit','retry-after','request-id'))}
                if response.ok:
                    raw = response.json(); assert isinstance(raw, dict)
                    log['response'] = raw
                else:
                    try: log['provider_error'] = response.json().get('error', {})
                    except ValueError: log['provider_error'] = {'type': 'non_json_error'}
                    retry_wait = seconds(response.headers.get('retry-after')) or 60
            except (requests.RequestException, ValueError, AssertionError) as error:
                log['error_type'] = type(error).__name__
            log['finished_epoch'] = time.time(); append(self.log_path, log)
            if 'response' in log:
                row = self.completed_row(log, board, task, expected)
                earlier_models = {r.get('returned_model') for r in self.latest().values() if r['case_id'] != case_id and r.get('returned_model')}
                if earlier_models and row['returned_model'] not in earlier_models:
                    self.defer('Returned model version changed during collection', review=True)
                return row
            if http == 429:
                limit=rate_limit_details(spec,log)
                if limit['review']:
                    return self.defer(limit['reason'],review=True,scope=limit['scope'],retry_source=limit['source'])
                if limit['scope']=='daily' or limit['wait']>120 or retry==4:
                    return self.defer(limit['reason'],limit['wait'],scope=limit['scope'],retry_source=limit['source'])
                self.defer(limit['reason'],limit['wait'],scope=limit['scope'],retry_source=limit['source'])
            elif http not in [None,200,500,502,503,504,529]:
                return self.defer(f'Provider access/configuration error HTTP {http}', review=True)
            elif retry == 4: return self.defer('Repeated transport/server failure; inspect logs', review=True)
            else:
                time.sleep(min(30, 5*2**retry))
        return self.defer('Unexpected retry exhaustion', review=True)


In [5]:
def audit_model(collector):
    spec = collector.spec
    cases = {r.puzzle_id: (chess.Board(r.solver_fen), 'evaluation', r.official_first_move) for r in puzzles.itertuples(index=False)}
    cases.update({p: (b, t, None) for p, b, t in FIXTURES})
    logs = [r for r in records(collector.log_path) if 'response' in r]
    by_trial = {(r['trial_id'], r['case_id']): r for r in logs}
    latest = collector.latest(); checked = 0; rescored = []
    assert len({r['case_id'] for r in logs}) == len(logs), 'More than one saved successful response for a case'
    for row in latest.values():
        if row['status'] != 'completed': continue
        board, task, expected = cases[row['puzzle_id']]
        log = by_trial[(row['trial_id'], row['case_id'])]
        assert log['request'] == request_body(spec, board, task)
        value = normalized(spec, log['response'])
        if not value.get('choices'): value = {'choices': [{'message': {'content': ''}, 'finish_reason': 'blocked_or_empty'}]}
        recorded_scorer=score if row.get('scoring_version')==SCORING_VERSION else score_strict
        category, answer, usable = recorded_scorer(board, task, value, expected)
        assert (category, answer, usable, int(category == 'success')) == (row['failure_type'], row['answer'], row['usable_answer'], row['success'])
        strict_category, strict_answer, strict_usable=score_strict(board,task,value,expected)
        category,answer,usable=score(board,task,value,expected)
        moves=mentioned_moves(board,value['choices'][0].get('message',{}).get('content')) if task=='evaluation' else []
        rescored.append(dict(row,strict_success=int(strict_category=='success'),strict_failure_type=strict_category,
            strict_answer=strict_answer,success=int(category=='success'),failure_type=category,answer=answer,usable_answer=usable,
            scoring_version=SCORING_VERSION if task=='evaluation' else 'strict_capability',
            mentioned_legal_moves=' '.join(moves),multiple_legal_moves_mentioned=len(moves)>1))
        checked += 1
    assert checked == len(logs), 'An unanswered log must be recovered before analysis'
    versions = sorted({r['returned_model'] for r in latest.values() if r.get('returned_model')})
    assert len(versions) <= 1, 'Provider returned different model versions; review before analysis'
    evaluated = [r for r in rescored if r['task'] == 'evaluation' and r['status'] == 'completed']
    if evaluated:
        frame = puzzles.merge(pd.DataFrame(evaluated), on='puzzle_id', how='left', validate='one_to_one')
    else:
        frame = puzzles.copy()
        for column in ['case_id','trial_id','status','failure_type','answer','usable_answer','success','returned_model','total_tokens']: frame[column] = np.nan
    frame.to_csv(collector.folder/'results.csv', index=False)
    audit = {'passed': True, 'responses_verified': checked, 'research_responses': len(evaluated),
             'sample_sha256': sha(sample_file), 'results_sha256': sha(collector.folder/'results.csv'),
             'returned_models': versions, 'manifest_sha256': sha(OUT/'manifest.json'),
             'scoring_version':SCORING_VERSION,'scoring_amendment_sha256':sha(CFG/'scoring_amendment.json')}
    save(collector.folder/'audit.json', audit)
    return frame, audit

def provider_once(collector):
    proof_path = collector.folder/'resume_check.json'
    proof = json.loads(proof_path.read_text()) if proof_path.exists() else {}
    prefix_ids = list(puzzles.loc[puzzles.operational_check, 'puzzle_id'])
    latest = collector.latest()
    prefix_verified = proof.get('new_generation_requests_on_resume') == 0 and all(latest.get('evaluation:'+p, {}).get('status') == 'completed' for p in prefix_ids)
    capabilities = []; failure_streak = 0
    for puzzle_id, board, task in FIXTURES:
        row = collector.run_case(puzzle_id, board, task)
        capabilities.append({'puzzle_id': puzzle_id, 'task': task, **row})
        if row['status'] != 'completed': break
        failure_streak = failure_streak+1 if row['failure_type'] == 'generation_limit' else 0
        if failure_streak >= 2: break
    counts = {task: sum(r.get('success',0) == 1 and r['task'] == task for r in capabilities) for task in ['side','legal','mate']}
    passed = len(capabilities) == 8 and all(r['status'] == 'completed' and r.get('usable_answer') for r in capabilities) and counts['side'] == 2 and counts['legal'] == 2 and counts['mate'] >= 2
    strict_passed = bool(passed)
    diagnostic = []
    if collector.spec['provider']=='anthropic' and len(capabilities)==8 and all(r['status']=='completed' for r in capabilities):
        # Post-check operational amendment: capability-only casing/outer Markdown normalization.
        # I preserve original raw/outcome records and keep research scoring strictly unchanged.
        fixture_map={p:(b,t) for p,b,t in FIXTURES}
        for r in capabilities:
            board,task=fixture_map[r['puzzle_id']]
            raw_log=next(x for x in records(collector.log_path) if x.get('trial_id')==r['trial_id'] and 'response' in x)
            value=normalized(collector.spec,raw_log['response'])
            text=value['choices'][0]['message']['content'].strip()
            if text.startswith('**') and text.endswith('**'):text=text[2:-2].strip()
            if task=='side':text=text.lower()
            value['choices'][0]['message']['content']=text
            category,answer,usable=score(board,task,value)
            diagnostic.append({'puzzle_id':r['puzzle_id'],'task':task,'success':category=='success','usable':usable})
        dc={t:sum(r['success'] for r in diagnostic if r['task']==t) for t in ['side','legal','mate']}
        passed=all(r['usable'] for r in diagnostic) and dc['side']==2 and dc['legal']==2 and dc['mate']>=2
    gate = {'passed': bool(passed), 'strict_format_passed':strict_passed, 'n': len(capabilities), 'successes': counts,
            'capability_only_format_diagnostic':diagnostic,
            'research_scoring':'Original strict scorer; no formatting normalization for research answers'}
    save(collector.folder/'capability_gate.json', gate)
    pd.DataFrame(capabilities).to_csv(collector.folder/'capability.csv', index=False)
    stop = None
    if passed:
        selected = (puzzles if RUN_MODE == 'bulk' else puzzles.loc[puzzles.pilot]) if prefix_verified else puzzles.loc[puzzles.operational_check]
        for row in selected.itertuples(index=False):
            previous = collector.latest().get('evaluation:'+row.puzzle_id)
            result = collector.run_case(row.puzzle_id, chess.Board(row.solver_fen), expected=row.official_first_move)
            if result['status'] != 'completed': stop = result; break
            if previous is None and (row.collection_order % 10 == 0 or row.collection_order == 1): print(collector.name, 'saved', row.collection_order, '/1000', flush=True)
    elif capabilities and capabilities[-1]['status'] != 'completed': stop = capabilities[-1]
    frame, audit = audit_model(collector)
    research_n = int(frame.status.eq('completed').sum())
    warmup_complete = bool(frame.loc[frame.operational_check, 'status'].eq('completed').all())
    if passed and warmup_complete and not prefix_verified:
        before = {'requests': sha(collector.log_path), 'outcomes': sha(collector.outcome_path)}
        for row in puzzles.loc[puzzles.operational_check].itertuples(index=False):
            collector.run_case(row.puzzle_id, chess.Board(row.solver_fen), expected=row.official_first_move)
        assert before == {'requests': sha(collector.log_path), 'outcomes': sha(collector.outcome_path)}, 'Resume changed completed answers or issued new generation requests'
        save(proof_path, {'model': collector.name, 'training_cases': CONFIG['warmup_n'], 'new_generation_requests_on_resume': 0,
                         'requests_unchanged_on_resume': True, 'outcomes_unchanged_on_resume': True,
                         'test_cases_used': 0, 'prefix_ids': prefix_ids, 'hashes': before})
        prefix_verified = True
    summary = {'model_key': collector.name, 'label': collector.spec['label'], 'capability_passed': passed,
               'research_n': research_n, 'planned_n': 1000, 'training_n': int((frame.status.eq('completed') & frame.split.eq('train')).sum()),
               'test_n': int((frame.status.eq('completed') & frame.split.eq('test')).sum()),
               'warmup_complete': warmup_complete, 'prefix_verified': bool(prefix_verified), 'complete': research_n == 1000,
               'status': 'complete' if research_n == 1000 else stop.get('status') if stop else 'pilot_complete' if research_n>=CONFIG['pilot_n'] and RUN_MODE=='pilot' else 'operational_check_complete' if warmup_complete else 'capability_gate_failed',
               'pause': stop, 'audit': audit}
    save(collector.folder/'summary.json', summary)
    return summary

def run_provider(name, spec):
    if spec['provider']=='anthropic':
        assert RUN_MODE=='pilot' or (RUN_MODE=='bulk' and spec.get('bulk_authorized') is True), 'Claude bulk collection requires documented approval.'
    collector = Collector(name, spec)
    try:
        with provider_lock(collector.folder):
            if not collector.connect(): return {'model_key': name, 'status': 'access_blocked', 'research_n': 0, 'warmup_complete': False}
            while True:
                result = provider_once(collector)
                if RUN_MODE in ['pilot','bulk'] and result['status'] == 'operational_check_complete' and result['prefix_verified']:
                    continue  # I verified this provider's prefix; I can now advance it independently.
                if RUN_MODE not in ['pilot','bulk'] or not WAIT_FOR_QUOTA or result['complete'] or result['status'] != 'paused_for_quota': return result
                pause = result['pause']
                print(name, 'paused; next eligible UTC time:', pause.get('resume_utc'), flush=True)
                while time.time() < pause['resume_epoch']: time.sleep(max(0, min(30, pause['resume_epoch']-time.time())))
    finally: collector.session.close()


def offline_checks():
    # I simulate HTTP and time; no real API calls or long sleeps.
    import tempfile
    class FakeResponse:
        def __init__(self,status,body,headers=None):self.status_code,self.body,self.headers=status,body,headers or {};self.ok=status==200
        def json(self):return self.body
    class FakeSession:
        def __init__(self,responses):self.responses,self.calls=responses,0
        def post(self,*args,**kwargs):self.calls+=1;return self.responses.pop(0)
        def close(self):pass
    class Clock:
        def __init__(self):self.now=time.time();self.slept=0
        def time(self):return self.now
        def sleep(self,seconds):self.slept+=seconds;self.now+=seconds
    original_time=globals()['time'];clock=Clock();globals()['time']=clock
    try:
        with tempfile.TemporaryDirectory(prefix='offline_check_',dir=OUT) as temporary:
            assert Path(temporary).resolve().is_relative_to(OUT.resolve())
            def mock(label,name='gpt_oss'):
                c=Collector(name,dict(CONFIG['models'][name]));c.session.close()
                c.folder=Path(temporary)/label;c.folder.mkdir();c.log_path=c.folder/'requests.jsonl';c.outcome_path=c.folder/'outcomes.jsonl'
                c.old_logs=[];c.spec['minimum_pause_seconds']=0
                return c
            fake={'choices':[{'message':{'content':'e2e4'},'finish_reason':'stop'}],'model':'openai/gpt-oss-120b','usage':{'total_tokens':500}}
            c=mock('saved');c.session=FakeSession([FakeResponse(200,fake)])
            first=c.run_case('test1',start,expected='d2d4');assert first['success']==0
            before=sha(c.outcome_path);assert c.run_case('test1',start,expected='d2d4')==first and c.session.calls==1
            c.outcome_path.unlink();assert c.run_case('test1',start,expected='d2d4')==first and c.session.calls==1 and sha(c.outcome_path)==before
            c=mock('zero');c.session=FakeSession([FakeResponse(429,{'error':{'message':'Rate limited'}},{'x-ratelimit-limit-tokens':'0'})])
            assert c.run_case('zero1',start,expected='e2e4')['status']=='needs_review'
            for name in ['gpt_oss','gemini']:
                c=mock('advisory_'+name,name)
                c.old_logs=[{'trial_id':str(i),'retry':0,'finished_epoch':clock.time(),'http_status':200,'response':{'usage':{'total_tokens':10000000}}} for i in range(1100)]
                save(c.folder/'pause.json',{'reason':'Conservative rolling 24-hour budget','resume_epoch':clock.time()+86400})
                wait,reason,review=c.quota_wait(request_body(c.spec,start));assert wait==0 and not review
            c=mock('daily');c.session=FakeSession([FakeResponse(429,{'error':{'message':'Rate limit reached on tokens per day (TPD). Please try again in 2m10s.'}},{'retry-after':'130'})])
            result=c.run_case('daily1',start,expected='e2e4')
            assert result['scope']=='daily' and c.session.calls==1 and not c.latest() and 130<=result['resume_epoch']-clock.time()<=133
            c=mock('minute');c.session=FakeSession([FakeResponse(429,{'error':{'message':'Tokens per minute (TPM) exceeded'}},{'retry-after':'1'}),FakeResponse(200,fake)])
            assert c.run_case('minute1',start,expected='e2e4')['status']=='completed' and c.session.calls==2 and clock.slept>=3
            c=mock('google_daily','gemini');error={'details':[{'violations':[{'quotaMetric':'generate_content_requests_per_day','quotaValue':'500'}]},{'retryDelay':'45s'}]}
            c.session=FakeSession([FakeResponse(429,{'error':error})]);result=c.run_case('gday',start,expected='e2e4')
            assert result['scope']=='daily' and c.session.calls==1 and result['resume_epoch']>=next_pacific_reset(clock.time())
            c=mock('header_daily');c.old_logs=[{'finished_epoch':clock.time(),'http_status':200,'rate_limits':{'x-ratelimit-remaining-requests':'0','x-ratelimit-reset-requests':'1h'}}]
            c.session=FakeSession([]);result=c.run_case('hday',start,expected='e2e4');assert result['scope']=='daily' and c.session.calls==0
            monthly=rate_limit_details(CONFIG['models']['gpt_oss'],{'provider_error':{'message':'Monthly quota exceeded'}});assert monthly['review']
    finally:globals()['time']=original_time
    fake=lambda answer,finish='stop':{'choices':[{'message':{'content':answer},'finish_reason':finish}]}
    assert score_strict(start,'legal',fake('e4'))[1]=='e2e4'
    assert score_strict(start,'legal',fake('Move: e2e4'))[0]=='unparseable'
    assert score_strict(start,'legal',fake('e2e5'))[0]=='illegal_move'
    assert score_strict(start,'legal',fake('e2e4','length'))[0]=='generation_limit'
    assert score(mate,'mate',fake('g6g7'))[0]=='success'
    print('Offline checks passed: immutable answers, crash replay, advisory daily estimates, provider daily stops, minute retries, access errors and scoring; no API calls.')

offline_checks()

def anthropic_offline_checks():
    import tempfile
    spec=CONFIG['models']['claude_haiku']
    body=request_body(spec,start)
    assert body['thinking']=={'type':'adaptive'} and body['output_config']=={'effort':'low'}
    assert not any(k in body for k in ['tools','temperature','seed','top_p'])
    raw={'model':spec['model'],'content':[{'type':'thinking','thinking':'hidden'},{'type':'text','text':'e2e4'}],'stop_reason':'end_turn','usage':{'input_tokens':500,'output_tokens':1000}}
    assert score(start,'evaluation',normalized(spec,raw),'e2e4')[0]=='success'
    assert usage_tokens(spec,{'response':raw})==1500
    assert score(start,'evaluation',normalized(spec,dict(raw,stop_reason='max_tokens')),'e2e4')[0]=='success'
    assert score(start,'evaluation',normalized(spec,dict(raw,content=[])),'e2e4')[0]=='unparseable'
    with tempfile.TemporaryDirectory(dir=OUT) as temp:
        c=Collector('claude_haiku',spec);c.folder=Path(temp);c.log_path=c.folder/'requests.jsonl';c.outcome_path=c.folder/'outcomes.jsonl'
        log={'trial_id':'offline','case_id':'evaluation:offline','puzzle_id':'offline','task':'evaluation','request':body,'response':raw}
        append(c.log_path,log)
        class NoNetwork:
            def post(self,*args,**kwargs):raise AssertionError('Unexpected network request')
        c.session.close();c.session=NoNetwork()
        first=c.run_case('offline',start,expected='d2d4');assert first['success']==0
        before=sha(c.outcome_path);assert c.run_case('offline',start,expected='d2d4')==first
        assert sha(c.outcome_path)==before
    print('Anthropic adapter offline checks passed; no network calls.')
anthropic_offline_checks()

# I test tolerances and boundaries without looking up expected moves in hidden reasoning.
def tolerant_scoring_checks():
    def response(text,finish='stop'):return {'choices':[{'message':{'content':text},'finish_reason':finish}]}
    board=chess.Board()
    for text in ['**e2e4**','The best move is e2e4.','I choose `e4`.','E2E4','e2-e4','d2d4 or e2e4','Do not play e2e4']:
        assert score(board,'evaluation',response(text),'e2e4')[0]=='success'
    for text in ['some2e4word','xe2e4x','d2d4','The correct move is not given.']:
        assert score(board,'evaluation',response(text),'e2e4')[0]!='success'
    assert score(board,'evaluation',response('e2e4','length'),'e2e4')[0]=='success'
    assert score(board,'evaluation',response('e2e','length'),'e2e4')[0]=='generation_limit'
    assert score(board,'evaluation',response('**Nf3**'),'g1f3')[0]=='success'
    assert score(board,'evaluation',response('**e2e5**'),'e2e4')[0]!='success'
    castle=chess.Board('r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1')
    assert score(castle,'evaluation',response('Play **0-0**.'),'e1g1')[0]=='success'
    promotion=chess.Board('7k/P7/8/8/8/8/8/7K w - - 0 1')
    assert score(promotion,'evaluation',response('a8=Q+'),'a7a8q')[0]=='success'
    spec=CONFIG['models']['claude_haiku']
    raw={'content':[{'type':'thinking','thinking':'e2e4'},{'type':'text','text':'d2d4'}],'stop_reason':'end_turn'}
    assert score(board,'evaluation',normalized(spec,raw),'e2e4')[0]=='legal_incorrect'
    print('Tolerant scoring tests passed: wording, Markdown, SAN/UCI, boundaries, multiple mentions, truncation, castling, promotion and hidden-thinking exclusion.')
tolerant_scoring_checks()


Offline checks passed: immutable answers, crash replay, advisory daily estimates, provider daily stops, minute retries, access errors and scoring; no API calls.
Anthropic adapter offline checks passed; no network calls.
Tolerant scoring tests passed: wording, Markdown, SAN/UCI, boundaries, multiple mentions, truncation, castling, promotion and hidden-thinking exclusion.


## 3. I launch only when cooldown has ended

In [6]:
if RUN_MODE=='setup':
    print('SETUP ONLY: I sent no API or model-metadata requests. When ready, I set RUN_MODE="pilot" and run all cells.')
else:
    SECRETS=[(ROOT/s['key_file']).read_text(encoding='utf-8-sig').strip() for k,s in CONFIG['models'].items() if k in SELECTED_MODELS]
    assert all(s and len(s.splitlines())==1 for s in SECRETS)
    summary={}
    with ThreadPoolExecutor(max_workers=len(SELECTED_MODELS)) as pool:
        futures={pool.submit(run_provider,name,spec):name for name,spec in CONFIG['models'].items() if name in SELECTED_MODELS}
        for future in as_completed(futures):summary[futures[future]]=future.result()
    save(OUT/'claude_pilot_progress.json' if SELECTED_MODELS==['claude_haiku'] else OUT/'progress.json',summary)
    display(pd.DataFrame([{'Model':CONFIG['models'][k]['label'],'Status':v['status'],'Saved research answers':v['research_n'],'Pilot target':100,'Final target':1000} for k,v in summary.items()]))
assert all(sha(ROOT/p)==h for p,h in protected.items())


claude_haiku saved 110 /1000


claude_haiku saved 120 /1000


claude_haiku saved 130 /1000


claude_haiku saved 140 /1000


claude_haiku saved 150 /1000


...

claude_haiku saved 970 /1000


claude_haiku saved 980 /1000


claude_haiku saved 990 /1000


claude_haiku saved 1000 /1000


,Model,Status,Saved research answers,Pilot target,Final target
0,Claude Haiku 5.5,complete,1000,100,1000


## My next step

I run notebook 05 after both 100-case pilots finish. If protocol and settings remain fixed, these answers count toward the 1,000-puzzle study. I keep notebook 06 closed to final evaluation until both collections are complete.

## My completed pilot

I completed the same 100 training puzzles for both deployments: **GPT-OSS 35/100 (35%)**, **Gemini 53/100 (53%)**. Both capability gates, raw-response audits and 108-response cached replays passed. No held-out puzzle was requested.

The first GPT-OSS invocation stopped at my local rolling-token estimate. In a logged operational-only retry, I bypassed that daily estimate and Groq accepted all remaining 94 research requests with HTTP 200. I retained request settings, minute pacing, scoring, the sample and analysis code. The amended collector now uses provider-enforced daily limits; I have not launched bulk.

I froze and fitted the exploratory early checkpoint in notebook 05. I review it before separately starting the remaining 900 puzzles per model. Final prediction now uses XGBoost; the pilot CARTs remain descriptive. No bulk collection is running.

## Re-score saved answers without API calls

I preserve raw requests and original outcome logs. The current `results.csv` and audits are derived views using `solution_mention_v1`; original strict tables are preserved in `strict_scoring_checkpoint`. A correct move mentioned among alternatives or in a negation still counts under the requested any-mention rule.

In [7]:
# Offline rescoring: I never connect to a provider or generate answers in this cell.
scoring_rows=[]
for name,spec in CONFIG['models'].items():
    collector=Collector(name,spec)
    try:
        with provider_lock(collector.folder):
            frame,audit=audit_model(collector)
            summary_path=collector.folder/'summary.json'
            summary=json.loads(summary_path.read_text())
            summary['audit']=audit
            summary['scoring_version']=SCORING_VERSION
            save(summary_path,summary)
            done=frame.loc[frame.status.eq('completed')]
            pilot=done.loc[done.pilot]
            scoring_rows.append({'Model':spec['label'],'Completed':len(done),'Strict correct':int(done.strict_success.sum()),
                'Tolerant correct':int(done.success.sum()),'Tolerant accuracy':float(done.success.mean()),
                'Pilot strict correct':int(pilot.strict_success.sum()),'Pilot tolerant correct':int(pilot.success.sum()),
                'Pilot puzzles':len(pilot),'New successes':int((done.success>done.strict_success).sum()),
                'Multiple legal moves mentioned':int(done.multiple_legal_moves_mentioned.sum())})
    finally:collector.session.close()
scoring_summary=pd.DataFrame(scoring_rows)
scoring_summary.to_csv(OUT/'tolerant_scoring_summary.csv',index=False)
save(OUT/'tolerant_scoring_report.json',{'scoring_version':SCORING_VERSION,'new_api_requests':0,
    'amendment_sha256':sha(CFG/'scoring_amendment.json'),'models':scoring_rows})
display(scoring_summary[['Model','Completed','Strict correct','Tolerant correct']])
print('Matched 100-puzzle training pilot:')
display(scoring_summary[['Model','Pilot strict correct','Pilot tolerant correct']])


,Model,Completed,Strict correct,Tolerant correct
0,GPT-OSS 120B,402,162,164
1,Gemini 3.5 Flash-Lite,1000,545,557
2,Claude Haiku 5.5,1000,286,490


Matched 100-puzzle training pilot:


,Model,Pilot strict correct,Pilot tolerant correct
0,GPT-OSS 120B,35,35
1,Gemini 3.5 Flash-Lite,53,54
2,Claude Haiku 5.5,28,46


## Claude pilot result and cost

I keep the original two-model early figures unchanged. This section reports Claude separately on the same 100 training puzzles. The initial capability gate had three exact-format failures; a disclosed capability-only diagnostic passed 8/8 after casing/outer-bold normalization. Research scoring now uses the documented tolerant solution-mention rule. I authorized Claude bulk collection after reviewing this pilot.

In [8]:
# I calculate cost from billed usage, including thinking, not just the visible move.
from collections import Counter
folder=OUT/'claude_haiku'
summary=json.loads((folder/'summary.json').read_text())
assert summary['research_n']>=100 and summary['audit']['passed']
research=pd.read_csv(folder/'results.csv',dtype={'puzzle_id':str}).loc[lambda d:d.status.eq('completed') & d.pilot].to_dict('records')
logs=records(folder/'requests.jsonl')
assert len(research)==100 and len({r['puzzle_id'] for r in research})==100
assert set(r['puzzle_id'] for r in research)==set(puzzles.loc[puzzles.pilot,'puzzle_id'])
prices=CONFIG['models']['claude_haiku']['pricing_usd_per_million']
def billed(group):
    rows=[r for r in logs if 'response' in r and ((r['task']=='evaluation' and r['puzzle_id'] in set(puzzles.loc[puzzles.pilot,'puzzle_id'])) if group=='pilot' else (r['task']!='evaluation'))]
    usage={k:sum(int(r['response'].get('usage',{}).get(k,0) or 0) for r in rows) for k in ['input_tokens','output_tokens','cache_read_input_tokens','cache_creation_input_tokens']}
    cost=(usage['input_tokens']*prices['input']+usage['output_tokens']*prices['output']+usage['cache_read_input_tokens']*prices['cache_read']+usage['cache_creation_input_tokens']*prices['cache_write_5m'])/1e6
    return {'successful_requests':len(rows),'usage':usage,'estimated_usd':cost}
pilot=billed('pilot');capability=billed('capability')
report={'model':CONFIG['models']['claude_haiku']['model'],'n':100,'correct':int(sum(r['success'] for r in research)),
        'accuracy':sum(r['success'] for r in research)/100,'scoring_version':SCORING_VERSION,'strict_correct':sum(r['strict_success'] for r in research),'training_n':100,'test_n':0,
        'failure_counts':dict(Counter(r['failure_type'] for r in research)),
        'capability_gate':json.loads((folder/'capability_gate.json').read_text()),
        'pilot_billing':pilot,'capability_billing':capability,
        'estimated_total_usd':pilot['estimated_usd']+capability['estimated_usd'],
        'projected_remaining_900_usd':9*pilot['estimated_usd'],'projected_1000_research_usd':10*pilot['estimated_usd'],
        'pricing_source':'https://platform.claude.com/docs/en/models/haiku-5-5/overview',
        'cost_note':'Standard global API list-price estimate from actual usage, including billed thinking; excludes taxes, credits and adjustments. Projections assume the pilot mean persists.',
        'bulk_started':False,'audit':summary['audit']}
if RUN_MODE=='pilot':save(folder/'pilot_report.json',report)
else:
    frozen=json.loads((folder/'pilot_report.json').read_text())
    assert frozen['n']==100 and frozen['correct']==report['correct']
    assert frozen['pilot_billing']==report['pilot_billing']
    print('I preserved the completed pilot report while bulk collection advanced.')
print(f"Claude Haiku 5.5: {report['correct']}/100 correct ({report['accuracy']:.0%}) under tolerant solution-mention scoring.")
print(f"Pilot ${pilot['estimated_usd']:.6f}; capability checks ${capability['estimated_usd']:.6f}; total ${report['estimated_total_usd']:.6f}.")
print(f"Projected remaining 900: ${report['projected_remaining_900_usd']:.3f}; pilot-based estimate.")
display(pd.DataFrame([{'Category':k,'Answers':v} for k,v in report['failure_counts'].items()]))


I preserved the completed pilot report while bulk collection advanced.
Claude Haiku 5.5: 46/100 correct (46%) under tolerant solution-mention scoring.
Pilot $0.057744; capability checks $0.001195; total $0.058939.
Projected remaining 900: $0.520; pilot-based estimate.


,Category,Answers
0,legal_incorrect,54
1,success,46
